In [ ]:
# ── Cell 0: Install dependencies & mount Drive ──────────────────────────────
!pip install -q wfdb tensorly scikit-learn xgboost imbalanced-learn
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os, json, time, logging, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Dict, Any
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.model_selection import StratifiedGroupKFold
from imblearn.over_sampling import SMOTE
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()],
    force=True,
)
logger = logging.getLogger(__name__)

def get_device():
    if torch.cuda.is_available():
        d = torch.device("cuda")
        logger.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        d = torch.device("cpu")
        logger.info("Using CPU")
    return d

device = get_device()
print("Imports done.")

In [ ]:
# ── Cell 2: ModelsConfig ─────────────────────────────────────────────────────
@dataclass
class ModelsConfig:
    # Point at results_v3 (preprocessed/filtered data from extraction v2)
    results_dir:   str = "/content/drive/MyDrive/results_v3"
    n_leads:       int = 12
    signal_len:    int = 5000
    n_classes:     int = 5
    random_seed:   int = 42
    train_frac:    float = 0.70
    val_frac:      float = 0.15
    # Deep model training
    batch_size:    int = 64
    num_epochs:    int = 70
    learning_rate: float = 3e-4
    weight_decay:  float = 1e-4
    patience:      int = 12
    def __post_init__(self):
        Path(self.results_dir).mkdir(parents=True, exist_ok=True)

config = ModelsConfig()

def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config.random_seed)
print("Config ready.")

In [ ]:
# ── Cell 3: CheckpointManager ────────────────────────────────────────────────
class CheckpointManager:
    def __init__(self, results_dir: str):
        self.dir = results_dir
        os.makedirs(results_dir, exist_ok=True)

    def _path(self, name: str) -> str:
        return os.path.join(self.dir, name)

    def load_master(self) -> dict:
        p = self._path("models_master.json")
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)
        return {"completed_models": []}

    def save_master(self, state: dict):
        with open(self._path("models_master.json"), "w") as f:
            json.dump(state, f, indent=2)

    def save_epoch_ckpt(self, model_name, epoch, model_state, opt_state, best_f1, no_improve):
        torch.save({
            "epoch": epoch, "model_state": model_state, "opt_state": opt_state,
            "best_f1": best_f1, "no_improve": no_improve,
        }, self._path(f"{model_name}_epoch_ckpt.pt"))

    def load_epoch_ckpt(self, model_name):
        p = self._path(f"{model_name}_epoch_ckpt.pt")
        return torch.load(p, map_location="cpu") if os.path.exists(p) else None

    def delete_epoch_ckpt(self, model_name):
        p = self._path(f"{model_name}_epoch_ckpt.pt")
        if os.path.exists(p): os.remove(p)

    def save_best_model(self, model_name, model_state):
        torch.save(model_state, self._path(f"{model_name}_best.pt"))

    def load_best_model(self, model_name):
        p = self._path(f"{model_name}_best.pt")
        return torch.load(p, map_location="cpu") if os.path.exists(p) else None

ckpt_mgr = CheckpointManager(config.results_dir)
print("CheckpointManager ready.")

In [ ]:
# ── Cell 4: Data loaders & patient-level splits ──────────────────────────────
# KEY CHANGE vs v1: patient-level splitting instead of record-level.
# PTB-XL has 21,375 records from 18,610 patients — same patient can have
# multiple recordings. Record-level splits risk the same patient appearing
# in both train and test (optimistic bias). Patient-level splits are stricter
# and more honest, consistent with published benchmarks.
#
# Implementation: load patient_ids_12lead.npy (saved by extraction v2),
# group records by patient and shuffle the patient IDs (seed 42); the split is
# NOT stratified by class.

def load_labels_and_classmap(cfg):
    d = cfg.results_dir
    y_raw = np.load(os.path.join(d, "y_raw_12lead.npy"))
    with open(os.path.join(d, "class_map.json")) as f:
        class_map = json.load(f)
    class_names = sorted(class_map, key=class_map.get)
    return y_raw, class_map, class_names

def load_X_raw(cfg):
    logger.info("Loading X_raw_12lead.npy (~5.1GB) ...")
    X = np.load(os.path.join(cfg.results_dir, "X_raw_12lead.npy"))
    logger.info(f"X_raw: {X.shape}")
    return X

def make_patient_splits(y, cfg):
    """
    Patient-level 70/15/15 split (shuffled patient IDs, not stratified by class).
    Loads patient_ids from results_v3; falls back to record-level if missing.
    Returns train_idx, val_idx, test_idx as arrays of record indices.
    """
    pat_path = os.path.join(cfg.results_dir, "patient_ids_12lead.npy")
    if not os.path.exists(pat_path):
        logger.warning("patient_ids_12lead.npy not found — falling back to record-level split")
        from sklearn.model_selection import train_test_split
        idx = np.arange(len(y))
        tv_idx, test_idx = train_test_split(idx, test_size=0.15, stratify=y,
                                             random_state=cfg.random_seed)
        tr_idx, val_idx  = train_test_split(tv_idx, test_size=0.15/0.85, stratify=y[tv_idx],
                                              random_state=cfg.random_seed)
        return tr_idx, val_idx, test_idx

    pat_ids = np.load(pat_path)
    unique_pats = np.unique(pat_ids)
    np.random.seed(cfg.random_seed)
    np.random.shuffle(unique_pats)

    n_test = int(len(unique_pats) * 0.15)
    n_val  = int(len(unique_pats) * 0.15)
    test_pats  = set(unique_pats[:n_test])
    val_pats   = set(unique_pats[n_test:n_test + n_val])
    train_pats = set(unique_pats[n_test + n_val:])

    train_idx = np.where(np.isin(pat_ids, list(train_pats)))[0]
    val_idx   = np.where(np.isin(pat_ids, list(val_pats)))[0]
    test_idx  = np.where(np.isin(pat_ids, list(test_pats)))[0]

    logger.info(f"Patient-level split: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")
    logger.info(f"  Train patients: {len(train_pats)} | Val: {len(val_pats)} | Test: {len(test_pats)}")
    return train_idx, val_idx, test_idx

In [ ]:
# ── Cell 5: Shared training utilities ───────────────────────────────────────

def augment_ecg_signal(x: torch.Tensor) -> torch.Tensor:
    """
    Augmentation for raw 12-lead ECG (12, L).
    v2 additions vs v1:
      - Baseline wander augmentation: adds a slow sinusoidal drift to simulate
        respiratory or motion artifact. Frequency 0.05-0.5 Hz, random amplitude.
      - Random lead permutation: randomly shuffles a subset of leads.
        Forces model not to rely on fixed lead ordering.
      - (Existing) Gaussian noise, amplitude scaling, time shift, lead dropout.
    """
    L = x.shape[-1]
    fs = 500

    # Gaussian noise
    if torch.rand(1).item() < 0.5:
        x = x + torch.randn_like(x) * (0.01 * x.std())

    # Amplitude scaling
    if torch.rand(1).item() < 0.3:
        x = x * (0.9 + 0.2 * torch.rand(1).item())

    # Circular time shift
    if torch.rand(1).item() < 0.3:
        shift = int(torch.randint(-50, 51, (1,)).item())
        x = torch.roll(x, shifts=shift, dims=-1)

    # Random single-lead dropout
    if torch.rand(1).item() < 0.15:
        lead_idx = torch.randint(0, x.shape[0], (1,)).item()
        x = x.clone()
        x[lead_idx] = 0.0

    # Baseline wander: slow sinusoidal drift (0.05-0.5 Hz)
    if torch.rand(1).item() < 0.4:
        freq = 0.05 + 0.45 * torch.rand(1).item()
        amp  = 0.05 + 0.15 * torch.rand(1).item()
        t    = torch.linspace(0, L / fs, L, device=x.device)
        phase = 2 * 3.14159 * torch.rand(1).item()
        wander = amp * torch.sin(2 * 3.14159 * freq * t + phase)
        x = x + wander.unsqueeze(0)

    # Random lead permutation (permute limb leads I,II,III,aVR,aVL,aVF = leads 0-5)
    if torch.rand(1).item() < 0.2:
        perm = torch.randperm(6)
        x = x.clone()
        x[:6] = x[perm]

    return x

class ECGRawDataset(Dataset):
    """12-lead raw signal dataset. Zero-copy view of full array + index list."""
    def __init__(self, X: np.ndarray, y: np.ndarray, indices: np.ndarray, augment: bool = False):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).long()
        self.indices = indices
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        x = self.X[idx].clone().float()
        if self.augment:
            x = augment_ecg_signal(x)
        return x, self.y[idx].clone()

class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al.) with optional label smoothing.
    Label smoothing (eps=0.1) slightly softens the one-hot targets, which
    prevents overconfident predictions and improves calibration — especially
    useful here since PTB-XL labels are based on physician likelihood scores
    rather than hard binary labels.
    gamma=1.0 with no alpha (removing alpha avoids the NORM collapse observed
    in ablation with alpha+gamma=2.0 combination).
    """
    def __init__(self, gamma: float = 1.0, label_smoothing: float = 0.1, n_classes: int = 5):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.n_classes = n_classes

    def forward(self, logits, targets):
        # Apply label smoothing
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits).scatter_(
                1, targets.unsqueeze(1), 1.0
            )
            smooth_targets = smooth_targets * (1 - self.label_smoothing) +                              self.label_smoothing / self.n_classes
        log_probs = torch.nn.functional.log_softmax(logits, dim=1)
        ce = -(smooth_targets * log_probs).sum(dim=1)
        pt = torch.exp(-torch.nn.functional.cross_entropy(logits, targets, reduction='none'))
        return ((1 - pt) ** self.gamma * ce).mean()

def make_weighted_sampler(y_train: np.ndarray) -> WeightedRandomSampler:
    classes, counts = np.unique(y_train, return_counts=True)
    weights_per_class = 1.0 / counts
    weights = np.array([weights_per_class[np.where(classes == c)[0][0]] for c in y_train])
    return WeightedRandomSampler(
        weights=torch.tensor(weights, dtype=torch.float),
        num_samples=len(weights),
        replacement=True,
    )

def compute_class_weights(y_train: np.ndarray, n_classes: int) -> torch.Tensor:
    counts = np.bincount(y_train, minlength=n_classes).astype(float)
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)

def evaluate(model, loader, criterion, dev):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(dev), yb.to(dev)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(yb)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
    n = len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / n, acc, f1, all_preds, all_labels

def save_confusion_matrix(preds, labels, class_names, save_path):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix')
    plt.tight_layout(); fig.savefig(save_path, dpi=150); plt.close(fig)
    logger.info(f"Confusion matrix saved: {save_path}")

In [ ]:
# ── Cell 6: train_deep_model ─────────────────────────────────────────────────
def train_deep_model(model_name, model, train_loader, val_loader, cfg, dev, ckpt,
                     class_weights=None, lr_override=None):
    """
    Train with epoch-level checkpointing. Identical to v1 except:
    - FocalLoss now includes label_smoothing=0.1
    - augment_ecg_signal has baseline wander + lead permutation
    """
    criterion = FocalLoss(gamma=1.0, label_smoothing=0.1, n_classes=cfg.n_classes)
    lr = lr_override if lr_override is not None else cfg.learning_rate
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.num_epochs)
    start_epoch, best_f1, no_improve, best_state = 0, 0.0, 0, None

    model.to(dev)
    saved = ckpt.load_epoch_ckpt(model_name)
    if saved:
        model.load_state_dict(saved["model_state"])
        optimizer.load_state_dict(saved["opt_state"])
        for state in optimizer.state.values():
            for k, v in state.items():
                if torch.is_tensor(v): state[k] = v.to(dev)
        start_epoch = saved["epoch"] + 1
        best_f1     = saved["best_f1"]
        no_improve  = saved["no_improve"]
        best_ckpt   = ckpt.load_best_model(model_name)
        if best_ckpt: best_state = best_ckpt
        logger.info(f"Resumed {model_name} from epoch {start_epoch}, best_f1={best_f1:.4f}")
    else:
        logger.info(f"Starting {model_name} from scratch")

    for epoch in range(start_epoch, cfg.num_epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(yb)
        scheduler.step()
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, dev)
        train_loss_avg = epoch_loss / len(train_loader.dataset)
        lr_now = scheduler.get_last_lr()[0]
        logger.info(
            f"[{model_name}] Epoch {epoch+1:3d}/{cfg.num_epochs} | "
            f"train_loss {train_loss_avg:.4f} | val_loss {val_loss:.4f} | "
            f"val_acc {val_acc:.4f} | val_f1 {val_f1:.4f} | lr {lr_now:.2e}"
        )
        if val_f1 > best_f1:
            best_f1, no_improve = val_f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            ckpt.save_best_model(model_name, best_state)
        else:
            no_improve += 1
        ckpt.save_epoch_ckpt(model_name, epoch, model.state_dict(),
                              optimizer.state_dict(), best_f1, no_improve)
        if no_improve >= cfg.patience:
            logger.info(f"Early stopping at epoch {epoch+1}")
            break

    if best_state: model.load_state_dict(best_state)
    ckpt.delete_epoch_ckpt(model_name)
    logger.info(f"{model_name} training done. Best val F1: {best_f1:.4f}")
    return model, best_f1

In [ ]:
# ── Cell 7: M1 — Mamba SSM (Pure PyTorch) ───────────────────────────────────
# Architecture unchanged from v1. Preprocessing improvement (bandpass+notch)
# feeds cleaner signals into the SSM, which should help especially with
# baseline wander artifacts that could confuse the slow drift in SSM states.

class MambaBlock(nn.Module):
    def __init__(self, d_model=128, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(d_model * expand)
        self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                   padding=d_conv - 1, groups=self.d_inner, bias=True)
        self.x_proj   = nn.Linear(self.d_inner, d_state * 2 + self.d_inner, bias=False)
        self.dt_proj  = nn.Linear(self.d_inner, self.d_inner, bias=True)
        self.A_log    = nn.Parameter(
            torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)
                      .unsqueeze(0).expand(self.d_inner, -1)))
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.drop     = nn.Dropout(dropout)
        self.norm     = nn.LayerNorm(d_model)

    def ssm_scan(self, x, delta, B, C):
        A     = -torch.exp(self.A_log.float())
        delta = torch.nn.functional.softplus(delta)
        dA    = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
        dB    = delta.unsqueeze(-1) * B.unsqueeze(2)
        dBx   = dB * x.unsqueeze(-1)
        log_dA_cum = torch.cumsum(torch.log(dA.clamp(min=1e-8)), dim=1)
        dA_cum     = torch.exp(log_dA_cum)
        h = dA_cum * torch.cumsum(dBx / dA_cum.clamp(min=1e-8), dim=1)
        y = (h * C.unsqueeze(2)).sum(dim=-1)
        return y + x * self.D.unsqueeze(0).unsqueeze(0)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        _, L, _ = x.shape
        xz = self.in_proj(x)
        x_b, z = xz.chunk(2, dim=-1)
        x_b = self.conv1d(x_b.permute(0, 2, 1))[..., :L].permute(0, 2, 1)
        x_b = torch.nn.functional.silu(x_b)
        ssm_in = self.x_proj(x_b)
        B_ssm  = ssm_in[..., :self.d_state]
        C_ssm  = ssm_in[..., self.d_state:2 * self.d_state]
        delta  = self.dt_proj(ssm_in[..., 2 * self.d_state:])
        y = self.ssm_scan(x_b, delta, B_ssm, C_ssm)
        y = y * torch.nn.functional.silu(z)
        return self.drop(self.out_proj(y)) + residual

class MambaECG(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, d_model=128, n_layers=4, d_state=16, dropout=0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, d_model // 2, kernel_size=15, stride=4, padding=7, bias=False),
            nn.BatchNorm1d(d_model // 2), nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=4, padding=3, bias=False),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.blocks   = nn.ModuleList([
            MambaBlock(d_model=d_model, d_state=d_state, d_conv=4, expand=2, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model)
        self.head     = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(d_model * 2, 256),
            nn.GELU(), nn.Dropout(dropout / 2), nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.stem(x).permute(0, 2, 1)
        for block in self.blocks: x = block(x)
        x = self.norm_out(x)
        x = torch.cat([x.mean(dim=1), x.max(dim=1)[0]], dim=1)
        return self.head(x)

# ── Run M1 ────────────────────────────────────────────────────────────────────
model_name = "M1_MambaSSM"
logger.info("=" * 70)
logger.info("M1: Mamba SSM (Pure PyTorch, parallel scan) on 12-lead raw ECG")
logger.info("=" * 70)
y_raw, _, class_names = load_labels_and_classmap(config)
X_raw = load_X_raw(config)
train_idx, val_idx, test_idx = make_patient_splits(y_raw, config)
cw      = compute_class_weights(y_raw[train_idx], config.n_classes)
sampler = make_weighted_sampler(y_raw[train_idx])
train_loader = DataLoader(ECGRawDataset(X_raw, y_raw, train_idx, augment=True),
                          batch_size=32, sampler=sampler, num_workers=0, pin_memory=True)
val_loader   = DataLoader(ECGRawDataset(X_raw, y_raw, val_idx),
                          batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(ECGRawDataset(X_raw, y_raw, test_idx),
                          batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

model_m1 = MambaECG(n_leads=config.n_leads, n_classes=config.n_classes)
model_m1, best_val_f1_m1 = train_deep_model(
    model_name, model_m1, train_loader, val_loader, config, device, ckpt_mgr, cw, lr_override=2e-4)

criterion_eval = FocalLoss(gamma=1.0, label_smoothing=0.1, n_classes=config.n_classes)
_, test_acc, test_f1, preds, labels = evaluate(model_m1, test_loader, criterion_eval, device)
test_prec = precision_score(labels, preds, average="macro", zero_division=0)
test_rec  = recall_score(labels, preds, average="macro", zero_division=0)
logger.info(f"M1 TEST: acc={test_acc:.4f} f1={test_f1:.4f} prec={test_prec:.4f} rec={test_rec:.4f}")
logger.info("\n" + classification_report(labels, preds, target_names=class_names, zero_division=0))
save_confusion_matrix(preds, labels, class_names,
                      os.path.join(config.results_dir, "M1_MambaSSM_cm_test.png"))
results_m1 = dict(model_name=model_name, test_accuracy=test_acc, test_f1=test_f1,
                  test_precision=test_prec, test_recall=test_rec, val_f1=best_val_f1_m1)
with open(os.path.join(config.results_dir, "M1_MambaSSM_results.json"), "w") as f:
    json.dump(results_m1, f, indent=2)
master = ckpt_mgr.load_master()
if "M1" not in master["completed_models"]:
    master["completed_models"].append("M1"); ckpt_mgr.save_master(master)
del X_raw, train_loader, val_loader, test_loader
import gc; gc.collect()
logger.info("M1 done.")

In [ ]:
# ── Cell 8: M2 — Multi-Lead 1D ResNet CNN ────────────────────────────────────
# Architecture unchanged from v1. Patient-level splits + improved augmentation.

class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=1, dropout=0.2):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.relu  = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        if self.downsample: identity = self.downsample(x)
        return self.relu(out + identity)

class MultiLead1DCNN(nn.Module):
    def __init__(self, n_leads=12, n_classes=5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, 64, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = nn.Sequential(ResidualBlock1D(64,  64),  ResidualBlock1D(64,  64))
        self.layer2 = nn.Sequential(ResidualBlock1D(64,  128, stride=2), ResidualBlock1D(128, 128))
        self.layer3 = nn.Sequential(ResidualBlock1D(128, 256, stride=2), ResidualBlock1D(256, 256))
        self.layer4 = nn.Sequential(ResidualBlock1D(256, 512, stride=2), ResidualBlock1D(512, 512))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(256, n_classes),
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.head(self.pool(x).squeeze(-1))

def run_m2_1dcnn(cfg, dev, ckpt, class_names):
    model_name = "M2_MultiLead1DCNN"
    logger.info("=" * 70); logger.info("M2: Multi-Lead 1D CNN"); logger.info("=" * 70)
    y, _, _ = load_labels_and_classmap(cfg)
    X_raw   = load_X_raw(cfg)
    train_idx, val_idx, test_idx = make_patient_splits(y, cfg)
    cw = compute_class_weights(y[train_idx], cfg.n_classes)
    sampler = make_weighted_sampler(y[train_idx])
    train_loader = DataLoader(ECGRawDataset(X_raw, y, train_idx, augment=True),
                              batch_size=cfg.batch_size, sampler=sampler, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(ECGRawDataset(X_raw, y, val_idx),
                              batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(ECGRawDataset(X_raw, y, test_idx),
                              batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    model = MultiLead1DCNN(n_leads=cfg.n_leads, n_classes=cfg.n_classes)
    model, best_val_f1 = train_deep_model(model_name, model, train_loader, val_loader, cfg, dev, ckpt, cw)
    criterion = FocalLoss(gamma=1.0, label_smoothing=0.1, n_classes=cfg.n_classes)
    _, test_acc, test_f1, preds, labels = evaluate(model, test_loader, criterion, dev)
    test_prec = precision_score(labels, preds, average="macro", zero_division=0)
    test_rec  = recall_score(labels, preds, average="macro", zero_division=0)
    logger.info(f"M2 TEST: acc={test_acc:.4f} f1={test_f1:.4f} prec={test_prec:.4f} rec={test_rec:.4f}")
    logger.info("\n" + classification_report(labels, preds, target_names=class_names, zero_division=0))
    save_confusion_matrix(preds, labels, class_names,
                          os.path.join(cfg.results_dir, "M2_MultiLead1DCNN_cm_test.png"))
    results = dict(model_name=model_name, test_accuracy=test_acc, test_f1=test_f1,
                   test_precision=test_prec, test_recall=test_rec, val_f1=best_val_f1)
    with open(os.path.join(cfg.results_dir, "M2_MultiLead1DCNN_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    del X_raw, train_loader, val_loader, test_loader; import gc; gc.collect()
    return results

In [ ]:
# ── Cell 9: M3 — Multi-Lead BiLSTM ──────────────────────────────────────────

class MultiLeadBiLSTM(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, hidden=192, num_layers=2, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, 64, kernel_size=15, stride=4, padding=7, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Conv1d(64, 96, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(96), nn.ReLU(inplace=True),
            nn.Conv1d(96, 96, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(96), nn.ReLU(inplace=True),
        )
        self.lstm = nn.LSTM(input_size=96, hidden_size=hidden, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden * 2 * 2, 128),
            nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(128, n_classes),
        )
    def forward(self, x):
        x = self.stem(x).permute(0, 2, 1)
        out, _ = self.lstm(x)
        pooled = torch.cat([out.mean(dim=1), out.max(dim=1)[0]], dim=1)
        return self.head(pooled)

def run_m3_bilstm(cfg, dev, ckpt, class_names):
    model_name = "M3_BiLSTM"
    logger.info("=" * 70); logger.info("M3: BiLSTM on 12-lead raw ECG"); logger.info("=" * 70)
    y, _, _ = load_labels_and_classmap(cfg)
    X_raw   = load_X_raw(cfg)
    train_idx, val_idx, test_idx = make_patient_splits(y, cfg)
    cw = compute_class_weights(y[train_idx], cfg.n_classes)
    sampler = make_weighted_sampler(y[train_idx])
    train_loader = DataLoader(ECGRawDataset(X_raw, y, train_idx, augment=True),
                              batch_size=cfg.batch_size, sampler=sampler, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(ECGRawDataset(X_raw, y, val_idx),
                              batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(ECGRawDataset(X_raw, y, test_idx),
                              batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    model = MultiLeadBiLSTM(n_leads=cfg.n_leads, n_classes=cfg.n_classes)
    model, best_val_f1 = train_deep_model(model_name, model, train_loader, val_loader, cfg, dev, ckpt, cw,
                                          lr_override=5e-4)
    criterion = FocalLoss(gamma=1.0, label_smoothing=0.1, n_classes=cfg.n_classes)
    _, test_acc, test_f1, preds, labels = evaluate(model, test_loader, criterion, dev)
    test_prec = precision_score(labels, preds, average="macro", zero_division=0)
    test_rec  = recall_score(labels, preds, average="macro", zero_division=0)
    logger.info(f"M3 TEST: acc={test_acc:.4f} f1={test_f1:.4f} prec={test_prec:.4f} rec={test_rec:.4f}")
    logger.info("\n" + classification_report(labels, preds, target_names=class_names, zero_division=0))
    save_confusion_matrix(preds, labels, class_names,
                          os.path.join(cfg.results_dir, "M3_BiLSTM_cm_test.png"))
    results = dict(model_name=model_name, test_accuracy=test_acc, test_f1=test_f1,
                   test_precision=test_prec, test_recall=test_rec, val_f1=best_val_f1)
    with open(os.path.join(cfg.results_dir, "M3_BiLSTM_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    del X_raw; import gc; gc.collect()
    return results

In [ ]:
# ── Cell 10: M4 — CRNN with Attention ───────────────────────────────────────
# (was M5_CRNN in v1 notebooks; now M4 since ResNet18-scalogram is dropped)

class CRNN(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, conv_ch=128, lstm_hidden=256,
                 n_lstm_layers=2, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_leads, 64,      kernel_size=11, stride=2, padding=5, bias=False),
            nn.BatchNorm1d(64),  nn.ReLU(inplace=True),
            nn.Conv1d(64,  conv_ch,     kernel_size=7,  stride=2, padding=3, bias=False),
            nn.BatchNorm1d(conv_ch), nn.ReLU(inplace=True),
            nn.Conv1d(conv_ch, conv_ch, kernel_size=5,  stride=2, padding=2, bias=False),
            nn.BatchNorm1d(conv_ch), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(input_size=conv_ch, hidden_size=lstm_hidden, num_layers=n_lstm_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = nn.Linear(lstm_hidden * 2, 1)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(inplace=True), nn.Linear(128, n_classes),
        )
    def forward(self, x):
        x   = self.conv(x).permute(0, 2, 1)
        out, _ = self.lstm(x)
        attn_w = torch.softmax(self.attn(out), dim=1)
        x = (out * attn_w).sum(dim=1)
        return self.head(x)

def run_m4_crnn(cfg, dev, ckpt, class_names):
    model_name = "M4_CRNN"
    logger.info("=" * 70); logger.info("M4: CRNN (Conv1D + BiLSTM + Attention)"); logger.info("=" * 70)
    y, _, _ = load_labels_and_classmap(cfg)
    X_raw   = load_X_raw(cfg)
    train_idx, val_idx, test_idx = make_patient_splits(y, cfg)
    cw = compute_class_weights(y[train_idx], cfg.n_classes)
    sampler = make_weighted_sampler(y[train_idx])
    train_loader = DataLoader(ECGRawDataset(X_raw, y, train_idx, augment=True),
                              batch_size=cfg.batch_size, sampler=sampler, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(ECGRawDataset(X_raw, y, val_idx),
                              batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(ECGRawDataset(X_raw, y, test_idx),
                              batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    model = CRNN(n_leads=cfg.n_leads, n_classes=cfg.n_classes)
    model, best_val_f1 = train_deep_model(model_name, model, train_loader, val_loader, cfg, dev, ckpt, cw,
                                          lr_override=4e-4)
    criterion = FocalLoss(gamma=1.0, label_smoothing=0.1, n_classes=cfg.n_classes)
    _, test_acc, test_f1, preds, labels = evaluate(model, test_loader, criterion, dev)
    test_prec = precision_score(labels, preds, average="macro", zero_division=0)
    test_rec  = recall_score(labels, preds, average="macro", zero_division=0)
    logger.info(f"M4 TEST: acc={test_acc:.4f} f1={test_f1:.4f} prec={test_prec:.4f} rec={test_rec:.4f}")
    logger.info("\n" + classification_report(labels, preds, target_names=class_names, zero_division=0))
    save_confusion_matrix(preds, labels, class_names,
                          os.path.join(cfg.results_dir, "M4_CRNN_cm_test.png"))
    results = dict(model_name=model_name, test_accuracy=test_acc, test_f1=test_f1,
                   test_precision=test_prec, test_recall=test_rec, val_f1=best_val_f1)
    with open(os.path.join(cfg.results_dir, "M4_CRNN_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    del X_raw; import gc; gc.collect()
    return results

In [ ]:
# ── Cell 11: M5 — Soft-Voting Ensemble (M2 + M3 + M4) ───────────────────────
# All three models trained on identical patient-level splits (same seed),
# so their test_idx arrays are consistent. Soft-voting averages softmax probs.

def run_m5_ensemble(cfg, dev, class_names):
    model_name = "M5_Ensemble_M2_M3_M4"
    logger.info("=" * 70)
    logger.info("M5: Ensemble (M2 1D-CNN + M3 BiLSTM + M4 CRNN)")
    logger.info("=" * 70)
    y, _, _ = load_labels_and_classmap(cfg)
    X_raw   = load_X_raw(cfg)
    train_idx, val_idx, test_idx = make_patient_splits(y, cfg)
    test_loader = DataLoader(ECGRawDataset(X_raw, y, test_idx),
                             batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)

    def load_model(cls, name, **kwargs):
        m = cls(**kwargs)
        state = ckpt_mgr.load_best_model(name)
        if state is None:
            raise FileNotFoundError(f"No saved best weights found for {name}")
        m.load_state_dict(state); m.to(dev); m.eval()
        return m

    m2 = load_model(MultiLead1DCNN,  "M2_MultiLead1DCNN", n_leads=cfg.n_leads, n_classes=cfg.n_classes)
    m3 = load_model(MultiLeadBiLSTM, "M3_BiLSTM",         n_leads=cfg.n_leads, n_classes=cfg.n_classes)
    m4 = load_model(CRNN,            "M4_CRNN",            n_leads=cfg.n_leads, n_classes=cfg.n_classes)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(dev)
            p2 = torch.softmax(m2(xb), dim=1)
            p3 = torch.softmax(m3(xb), dim=1)
            p4 = torch.softmax(m4(xb), dim=1)
            avg = (p2 + p3 + p4) / 3.0
            all_preds.extend(avg.argmax(dim=1).cpu().numpy())
            all_labels.extend(yb.numpy())

    test_acc  = accuracy_score(all_labels, all_preds)
    test_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    test_prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    test_rec  = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    logger.info(f"M5 ENSEMBLE TEST: acc={test_acc:.4f} f1={test_f1:.4f} prec={test_prec:.4f} rec={test_rec:.4f}")
    logger.info("\n" + classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))
    save_confusion_matrix(all_preds, all_labels, class_names,
                          os.path.join(cfg.results_dir, "M5_Ensemble_cm_test.png"))
    results = dict(model_name=model_name, test_accuracy=test_acc, test_f1=test_f1,
                   test_precision=test_prec, test_recall=test_rec)
    with open(os.path.join(cfg.results_dir, "M5_Ensemble_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    del X_raw; import gc; gc.collect()
    return results

In [ ]:
# ── Cell 13: MAIN — run all models in sequence ───────────────────────────────
# Run this cell to execute M1-M5 in order. M6 (MiniRocket + LightGBM) is run in
# 02_revision_analysis.ipynb. Each model loads/frees X_raw
# independently to avoid RAM exhaustion.

def main_models():
    cfg = ModelsConfig()
    set_seed(cfg.random_seed)
    dev  = get_device()
    ckpt = CheckpointManager(cfg.results_dir)
    master = ckpt.load_master()
    y_raw, class_map, class_names = load_labels_and_classmap(cfg)
    logger.info(f"Classes: {class_map}")
    all_results = {}

    def run_and_track(key, label, fn, *args):
        if key not in master["completed_models"]:
            logger.info(f"\nStarting {label}")
            result = fn(*args)
            all_results[key] = result
            master["completed_models"].append(key)
            ckpt.save_master(master)
        else:
            logger.info(f"{label} already complete, loading JSON")
            json_path = os.path.join(cfg.results_dir, f"{key}_results.json")
            with open(json_path) as f: all_results[key] = json.load(f)

    run_and_track("M1_MambaSSM",      "M1: Mamba SSM",       lambda: None)  # M1 is standalone Cell 7
    run_and_track("M2_MultiLead1DCNN","M2: 1D CNN",          run_m2_1dcnn,  cfg, dev, ckpt, class_names)
    run_and_track("M3_BiLSTM",        "M3: BiLSTM",          run_m3_bilstm, cfg, dev, ckpt, class_names)
    run_and_track("M4_CRNN",          "M4: CRNN",            run_m4_crnn,   cfg, dev, ckpt, class_names)
    run_and_track("M5_Ensemble",      "M5: Ensemble",        run_m5_ensemble, cfg, dev, class_names)

    # Print summary
    rows = []
    for mname, res in all_results.items():
        if isinstance(res, dict):
            rows.append({"Model": mname,
                         "Acc": f"{res.get('test_accuracy', float('nan')):.4f}",
                         "F1":  f"{res.get('test_f1', float('nan')):.4f}",
                         "Prec":f"{res.get('test_precision', float('nan')):.4f}",
                         "Rec": f"{res.get('test_recall', float('nan')):.4f}"})
    summary = pd.DataFrame(rows)
    csv_path = os.path.join(cfg.results_dir, "all_models_summary.csv")
    summary.to_csv(csv_path, index=False)
    logger.info("\n" + "=" * 70)
    logger.info("ALL MODELS SUMMARY")
    logger.info("=" * 70)
    logger.info("\n" + summary.to_string(index=False))
    logger.info("\nEXPERIMENT COMPLETE")
    return all_results

In [ ]:
    main_models()